# Signed Jumps 01 — volatility engine + signed Hawkes jump layer (Binance spot 1m)

Protocol: `discussions/Signed_Jumps/001_claude_sj01_preregistration.md` (fixed before this run).

Ladder: (a) Poisson → (b) engine-modulated Poisson (null) → (c) excitation → (d) parent sign matters → (e) + jump-truncated skewness.
Gates on TEST (last 30% of days): **G1** (c)>(b), **G2a** (d)>(c), **G2b** (e)>(d) [Bonferroni], **G3** 15-minute signed jump forecast vs (b).

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj
importlib.reload(sj)
assert sj.VERSION.startswith('SJ01-0.1')
SYNTHETIC=os.getenv('SJ01_SYNTHETIC','0')=='1'
p=sj.Params()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj01-');OUT=Path(tmp.name)/'01'
    p=sj.Params(n_boot=200,half_lives=(3,10),min_test_jumps_per_sign=20,mc_paths=100,c_sensitivity=(4.5,))
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'01'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(sj.registration(p),indent=1)[:1800])

...........                                                              [100%]
11 passed in 46.83s


{
 "assets": [
  "BTC",
  "ETH",
  "SOL",
  "XRP"
 ],
 "c": 4.0,
 "c_sensitivity": [
  3.5,
  4.5,
  5.0
 ],
 "floor_bp": 5.0,
 "fourier_k": 4,
 "har_windows": [
  30,
  60,
  240,
  1440
 ],
 "har_target": 60,
 "min_cover": 0.75,
 "fast_half_life": 30,
 "slow_half_life": 240,
 "surprise_windows": [
  10,
  60
 ],
 "tsk_window": 60,
 "tsk_min": 45,
 "half_lives": [
  1,
  3,
  10,
  30,
  90
 ],
 "dev_fraction": 0.7,
 "n_boot": 2000,
 "horizons": [
  5,
  15,
  60
 ],
 "primary_horizon": 15,
 "origin_step": 15,
 "mc_paths": 200,
 "min_test_jumps_per_sign": 200,
 "status": "PREREGISTERED_SJ001",
 "version": "SJ01-0.1.0",
 "gates": {
  "G1_excitation": "(c) vs (b): TEST logLik/row delta > 0, day-bootstrap q2.5 > 0, >= min jumps per sign",
  "G2a_parent_sign": "(d) vs (c): same rule at Bonferroni q1.25",
  "G2b_truncated_skew": "(e) vs (d): same rule at Bonferroni q1.25",
  "G2": "PASS if

## Data and volatility engine

In [2]:
T0=time.time()
if SYNTHETIC:
    S={a:s for a,s in zip(p.assets,sj.simulate(n_days=40,n_assets=len(p.assets),eta=((0.2,0.0),(0.05,0.3)),seed=5,df=8).values())}
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={}
    for A in p.assets:
        f=BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet'
        if not f.exists():print('MISSING',f);continue
        S[A]=sj.load_spot_1m(f)
assert len(S)==len(p.assets),f'assets loaded: {list(S)}'
assets,pan,dev_end=sj.prepare(S,p)
print('DEV/TEST boundary:',dev_end,'| panel rows',len(pan.r),f'| {time.time()-T0:.0f}s')
q=[]
for i,a in enumerate(assets):
    m=pan.asset_idx==i
    r=a.r;ok=np.isfinite(r)
    row={'asset':a.name,'first':str(a.time[0]),'last':str(a.time[-1]),'grid_bars':len(r),'observed_returns':int(ok.sum()),
         'zero_return_share':float((r[ok]==0).mean()),'panel_rows':int(m.sum()),
         'HAR_coef':np.round(a.cols['har_coef'],3).tolist()}
    for s in sj.SIGNS:
        for per,mm in (('dev',pan.dev),('test',~pan.dev)):
            row[f'{s}_{per}']=int(pan.J[s][m&mm].sum())
        row[f'{s}_per_day']=float(pan.J[s][m].sum()/(m.sum()/1440))
    q.append(row)
quality=pd.DataFrame(q);display(quality);quality.to_csv(OUT/'01_data_quality.csv',index=False)
sz=[]
for i,a in enumerate(assets):
    m=pan.asset_idx==i
    for s in sj.SIGNS:
        rr=np.abs(pan.r[m&(pan.J[s]>0)])*1e4
        zz=np.abs(pan.Z[m&(pan.J[s]>0)])
        sz.append({'asset':a.name,'sign':s,'n':len(rr),'mean_bp':rr.mean(),'median_bp':np.median(rr),'p90_bp':np.quantile(rr,.9),
                   'mean_absZ':zz.mean()})
sizes=pd.DataFrame(sz);display(sizes.round(2));sizes.to_csv(OUT/'01_jump_sizes.csv',index=False)

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:172: RuntimeWarning: divide by zero encountered in log
  feats.append(np.log(m))
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: divide by zero encountered in divide
  Z = z / np.sqrt(sig2c)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: invalid value encountered in divide
  Z = z / np.sqrt(sig2c)


DEV/TEST boundary: 2026-02-15 00:00:00+00:00 | panel rows 4625542 | 96s


,asset,first,last,grid_bars,observed_returns,zero_return_share,panel_rows,HAR_coef,up_dev,up_test,up_per_day,down_dev,down_test,down_per_day
0,BTC,2024-08-02 00:00:00+00:00,2026-10-14 23:59:00+00:00,1157760,1157759,0.032411,1156376,"[0.082, 0.25, 0.247, 0.185, 0.229]",1673,975,3.297474,1653,870,3.141815
1,ETH,2024-08-02 00:00:00+00:00,2026-10-14 23:59:00+00:00,1157760,1157759,0.012308,1156334,"[0.085, 0.271, 0.166, 0.192, 0.262]",1641,961,3.240309,1676,870,3.170572
2,SOL,2024-08-02 00:00:00+00:00,2026-10-14 23:59:00+00:00,1157760,1157759,0.056112,1156468,"[0.056, 0.265, 0.204, 0.191, 0.242]",1240,757,2.486606,1169,758,2.399444
3,XRP,2024-08-02 00:00:00+00:00,2026-10-14 23:59:00+00:00,1157760,1157759,0.059008,1156364,"[0.008, 0.301, 0.204, 0.196, 0.235]",1431,853,2.844226,1339,714,2.556565


,asset,sign,n,mean_bp,median_bp,p90_bp,mean_absZ
0,BTC,up,2648,26.36,20.52,49.63,5.64
1,BTC,down,2523,26.30,21.40,48.52,5.59
2,ETH,up,2602,47.02,39.25,80.35,5.79
3,ETH,down,2546,48.86,39.39,81.98,5.76
4,SOL,up,1997,55.63,46.30,94.20,5.53
5,SOL,down,1927,55.33,44.20,92.52,5.55
6,XRP,up,2284,52.90,39.71,96.82,5.52
7,XRP,down,2053,54.25,38.47,98.57,5.52


## Fit the ladder on DEV

In [3]:
T1=time.time()
fits,Ecache=sj.fit_ladder(assets,pan,p)
print(f'fit {time.time()-T1:.0f}s')
fr=[]
for (m,s),f in fits.items():
    names=sj.design_names(pan,m,p)
    row={'model':m,'sign':s,'half_life_min':f.half_life,'ll_dev':f.ll_dev,'n_dev':f.n_dev,'converged':f.converged}
    row.update({n:v for n,v in zip(names,f.b)})
    for k,v in enumerate(f.eta):row[f'eta{k}']=v
    fr.append(row)
fit_tab=pd.DataFrame(fr);display(fit_tab.round(4));fit_tab.to_csv(OUT/'01_fits.csv',index=False)
assert fit_tab.converged.all() or SYNTHETIC,'non-converged fit'
eta=sj.eta_matrix(fits)
# cluster(day)-robust SEs on DEV for (d) and (e)
dev=pan.dev;se_rows=[]
for m in ('d','e'):
    X=sj.design(pan,m)[dev]
    for s in sj.SIGNS:
        f=fits[(m,s)];Ex=sj.excitation_design(Ecache[f.half_life],m,s)[dev]
        se=sj.cluster_se(f,X.astype(float),Ex,pan.J[s][dev],pan.day[dev])
        names=sj.design_names(pan,m,p)+['eta_from_up','eta_from_down']
        for n,v,e in zip(names,np.concatenate([f.b,f.eta]),se):
            se_rows.append({'model':m,'child_sign':s,'param':n,'estimate':v,'cluster_se':e})
se_tab=pd.DataFrame(se_rows);se_tab.to_csv(OUT/'01_param_se.csv',index=False)
display(se_tab[se_tab.param.str.startswith(('eta','TSK','L_','F_','TV'))].round(4))
cont=eta.from_up[0]+eta.from_down[1]-eta.from_down[0]-eta.from_up[1]
print('eta matrix (d):');display(eta.round(4))
print(f"branching (c) {sj.branching(fits,'c'):.3f} | (d) {sj.branching(fits,'d'):.3f} | continuation index {cont:+.3f}")

fit 423s


,model,sign,half_life_min,ll_dev,n_dev,converged,asset_BTC,asset_ETH,asset_SOL,asset_XRP,...,sin4,cos4,weekend,L_slow,F_fast_minus_slow,TV10,TV60,eta0,eta1,TSK
0,a,up,NaN,-43592.6963,3231841,True,-6.1789,-6.1982,-6.4787,-6.3352,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a,down,NaN,-42637.7544,3231841,True,-6.1909,-6.1770,-6.5377,-6.4017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,b,up,NaN,-42348.8812,3231841,True,-6.1469,-6.1404,-6.4652,-6.3035,...,0.0144,-0.0823,0.2459,-0.1919,0.0807,0.8667,0.1392,NaN,NaN,NaN
3,b,down,NaN,-41034.3463,3231841,True,-6.1944,-6.1460,-6.5511,-6.3930,...,-0.0033,-0.0332,0.2566,-0.1736,0.1881,0.9269,0.2332,NaN,NaN,NaN
4,c,up,1.0,-40780.7370,3231841,True,-6.2435,-6.2618,-6.5881,-6.4119,...,0.0261,-0.0797,0.2308,-0.2099,-0.2537,0.7522,0.4393,0.0828,NaN,NaN
5,c,down,1.0,-39790.2410,3231841,True,-6.3094,-6.2754,-6.6743,-6.5071,...,-0.0211,-0.0499,0.2779,-0.1987,-0.0103,0.8380,0.4435,0.0732,NaN,NaN
6,d,up,1.0,-40775.5911,3231841,True,-6.2433,-6.2619,-6.5879,-6.4119,...,0.0261,-0.0794,0.2303,-0.2093,-0.2535,0.7520,0.4409,0.0922,0.0734,NaN
7,d,down,1.0,-39700.9286,3231841,True,-6.3105,-6.2814,-6.6754,-6.5087,...,-0.0220,-0.0494,0.2820,-0.2002,-0.0142,0.8364,0.4414,0.0373,0.1120,NaN
8,e,up,1.0,-40711.6093,3231841,True,-6.2603,-6.2775,-6.6037,-6.4264,...,0.0248,-0.0795,0.2372,-0.2076,-0.2441,0.7525,0.4297,0.0912,0.0739,0.1589
9,e,down,1.0,-39645.6670,3231841,True,-6.3206,-6.2906,-6.6848,-6.5198,...,-0.0213,-0.0495,0.2745,-0.1973,0.0008,0.8342,0.4404,0.0377,0.1110,-0.1503


,model,child_sign,param,estimate,cluster_se
13,d,up,L_slow,-0.2093,0.0362
14,d,up,F_fast_minus_slow,-0.2535,0.0668
15,d,up,TV10,0.7520,0.0386
16,d,up,TV60,0.4409,0.0871
17,d,up,eta_from_up,0.0922,0.0062
18,d,up,eta_from_down,0.0734,0.0057
32,d,down,L_slow,-0.2002,0.0348
33,d,down,F_fast_minus_slow,-0.0142,0.0748
34,d,down,TV10,0.8364,0.0405
35,d,down,TV60,0.4414,0.0948


eta matrix (d):


,child,from_up,from_down,half_life_min
0,up,0.0922,0.0734,1
1,down,0.0373,0.1120,1


branching (c) 0.156 | (d) 0.155 | continuation index +0.094


## Gates G1, G2 on TEST

In [4]:
cmp,njump=sj.compare(fits,pan,Ecache,p)
print('TEST jumps:',njump);display(cmp[['test','delta_per_row','q0125','q025','q975','n_days','status','bonferroni']])
cmp.to_csv(OUT/'01_tests.csv',index=False)

TEST jumps: {'up': 3546, 'down': 3212}


,test,delta_per_row,q0125,q025,q975,n_days,status,bonferroni
0,b_vs_a,0.000730,0.000522,0.000548,0.000923,242,PASS,False
1,G1_c_vs_b,0.000829,0.000635,0.000662,0.001013,242,PASS,False
2,G2a_d_vs_c,0.000046,0.000023,0.000026,0.000067,242,PASS,True
3,G2b_e_vs_d,0.000069,0.000043,0.000046,0.000091,242,PASS,True
4,d_vs_b,0.000875,0.000672,0.000699,0.001069,242,PASS,False
5,e_vs_b,0.000945,0.000739,0.000764,0.001139,242,PASS,False


## G3: 15-minute signed jump forecast (5 and 60 secondary)

In [5]:
sel=sj.selected_model(cmp);print('selected model:',sel)
T2=time.time()
g3,cal=sj.g3_table(fits,sel,assets,pan,p)
print(f'{time.time()-T2:.0f}s');display(g3.round(5));g3.to_csv(OUT/'01_g3.csv',index=False)
if len(cal):display(cal.round(4));cal.to_csv(OUT/'01_sign_calibration.csv',index=False)

selected model: e
121s


,delta_per_row,q0125,q025,q975,q09875,n_days,n_rows,horizon_min,model,status,mean_p_up_b,mean_p_up_model,obs_up,mean_p_dn_b,mean_p_dn_model,obs_dn
0,0.00119,0.00063,0.00069,0.00170,0.00177,242,92913,5,e,PASS,0.00969,0.00932,0.01161,0.00926,0.00891,0.01025
1,0.00193,0.00131,0.00140,0.00245,0.00251,242,92909,15,e,PASS,0.02862,0.02660,0.03412,0.02730,0.02529,0.03065
2,0.00635,0.00483,0.00499,0.00775,0.00793,242,92900,60,e,PASS,0.10715,0.09889,0.12509,0.10156,0.09340,0.11367


,bin,n,pred_up_share,up_only,down_only,horizon_min
0,0,396,0.3868,0.3990,0.5606,5
1,1,396,0.4670,0.4773,0.4975,5
2,2,396,0.5208,0.5227,0.4571,5
3,3,396,0.5757,0.5808,0.4015,5
4,4,396,0.6559,0.6162,0.3586,5
5,0,1142,0.3949,0.3975,0.5306,15
6,1,1141,0.4743,0.4636,0.4873,15
7,2,1141,0.5247,0.5127,0.4382,15
8,3,1141,0.5745,0.5609,0.3839,15
9,4,1141,0.6517,0.5697,0.3821,15


## Sensitivity: jump threshold c (reported, never used for decisions)

In [6]:
sens=[]
from dataclasses import replace
for cc in p.c_sensitivity:
    pc=replace(p,c=cc)
    sj.rejump(assets,pc);pn=sj.build_panel(assets,pc)
    fc,Ec=sj.fit_ladder(assets,pn,pc,models=('a','b','c','d'))
    cm,nj=sj.compare(fc,pn,Ec,pc)
    for _,r in cm.iterrows():
        sens.append({'c':cc,'test':r.test,'delta_per_row':r.delta_per_row,'q025':r.q025,'status':r.status,
                     'jumps_up':nj['up'],'jumps_down':nj['down'],'rho_c':sj.branching(fc,'c'),'rho_d':sj.branching(fc,'d')})
sj.rejump(assets,p)
sens=pd.DataFrame(sens);display(sens.round(6));sens.to_csv(OUT/'01_c_sensitivity.csv',index=False)

,c,test,delta_per_row,q025,status,jumps_up,jumps_down,rho_c,rho_d
0,3.5,b_vs_a,0.000894,0.000706,PASS,5485,5154,0.187236,0.192276
1,3.5,G1_c_vs_b,0.001102,0.000918,PASS,5485,5154,0.187236,0.192276
2,3.5,G2a_d_vs_c,0.000077,0.000050,PASS,5485,5154,0.187236,0.192276
3,3.5,d_vs_b,0.001178,0.000978,PASS,5485,5154,0.187236,0.192276
4,4.5,b_vs_a,0.000631,0.000471,PASS,2395,2130,0.146835,0.146108
5,4.5,G1_c_vs_b,0.000630,0.000479,PASS,2395,2130,0.146835,0.146108
6,4.5,G2a_d_vs_c,0.000045,0.000025,PASS,2395,2130,0.146835,0.146108
7,4.5,d_vs_b,0.000675,0.000518,PASS,2395,2130,0.146835,0.146108
8,5.0,b_vs_a,0.000564,0.000412,PASS,1691,1487,0.145807,0.145849
9,5.0,G1_c_vs_b,0.000441,0.000321,PASS,1691,1487,0.145807,0.145849


## Decision

In [7]:
dec=sj.decide(cmp,g3,p)
dec.update(branching_c=sj.branching(fits,'c'),branching_d=sj.branching(fits,'d'),continuation_index=float(cont),
           eta_matrix=eta.to_dict(orient='records'),test_jumps=njump,dev_end=str(dev_end),synthetic=SYNTHETIC)
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'01_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'01_manifest.json').write_text(json.dumps({'registration':sj.registration(p),'runtime_s':time.time()-T0,
        'assets':list(S),'rows':int(len(pan.r))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

{
 "version": "SJ01-0.1.0",
 "status": "PREREGISTERED_SJ001",
 "G1_excitation": "PASS",
 "G2a_parent_sign": "PASS",
 "G2b_truncated_skew": "PASS",
 "G2": "PASS",
 "G3_horizon": "PASS",
 "G3_model": "e",
 "G4_build_SJ02": true,
 "branching_c": 0.1560213540879659,
 "branching_d": 0.15531827414735727,
 "continuation_index": 0.09352746542540316,
 "eta_matrix": [
  {
   "child": "up",
   "from_up": 0.09223335627188602,
   "from_down": 0.07338109036902508,
   "half_life_min": 1
  },
  {
   "child": "down",
   "from_up": 0.03727933281976928,
   "from_down": 0.11195453234231151,
   "half_life_min": 1
  }
 ],
 "test_jumps": {
  "up": 3546,
  "down": 3212
 },
 "dev_end": "2026-02-15 00:00:00+00:00",
 "synthetic": false
}
total 1566s
